In [1]:
import numpy as np
import pandas as pd
import os
from datetime import datetime, date
from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.tsa.vector_ar.vecm import VECM
import scipy.linalg as la
from scipy.linalg import eig
from statsmodels.tsa.vector_ar.var_model import VAR
import statsmodels.api as sm
print(f"modules have been imported.")

modules have been imported.


In [3]:
# Step 1: Lets grab the historical time series from Mohaddes first. 
# Question should we grab it by economies first? if yes then lets take the 'Country Data'
print(os.getcwd())
target_filename = "Country Data (1979Q2-2023Q3).xls"
target_file_path = os.getcwd() + "\\" + "GVAR Database (1979Q2-2023Q3)\\" + target_filename
print(f"target_file_path is: {target_file_path}")
# dataframe the country data
countries_df = pd.read_excel(target_file_path, sheet_name = None)
countries_df.keys()


C:\Users\benchee
target_file_path is: C:\Users\benchee\GVAR Database (1979Q2-2023Q3)\Country Data (1979Q2-2023Q3).xls


dict_keys(['ARGENTINA', 'AUSTRALIA', 'BRAZIL', 'CANADA', 'CHINA', 'CHILE', 'EURO', 'INDIA', 'INDONESIA', 'JAPAN', 'KOREA', 'MALAYSIA', 'MEXICO', 'NORWAY', 'NEW ZEALAND', 'PERU', 'PHILIPPINES', 'SOUTH AFRICA', 'SAUDI ARABIA', 'SINGAPORE', 'SWEDEN', 'SWITZERLAND', 'THAILAND', 'TURKEY', 'UNITED KINGDOM', 'USA'])

In [4]:
us_df = countries_df['USA']
# convert the date into datetime format 
# ie: 1979Q2 is 30-Jun-1979.
us_df['date_2'] = pd.PeriodIndex(us_df['date'], freq = 'Q').to_timestamp(how = "end").date

target_cols = [col for col in us_df.columns if 'date' not in col]
target_cols = ['date_2'] + target_cols 
print(target_cols)
us_df = us_df[target_cols]
us_df = us_df.rename(columns={'date_2': 'date'})
us_df

['date_2', 'y', 'Dp', 'eq', 'r', 'lr', 'ys', 'Dps', 'eps', 'poil', 'pmat', 'pmetal']


,date,y,Dp,eq,r,lr,ys,Dps,eps,poil,pmat,pmetal
0,1979-06-30,3.960503,0.033730,0.684670,0.022407,0.021804,3.723439,0.027880,-2.381442,3.433544,4.148960,4.200168
1,1979-09-30,3.967677,0.032477,0.699070,0.023084,0.021781,3.736803,0.029817,-2.420323,3.576322,4.272871,4.198063
2,1979-12-31,3.970622,0.028823,0.640953,0.027982,0.024841,3.755427,0.029535,-2.440535,3.696729,4.220098,4.305733
3,1980-03-31,3.973806,0.038217,0.615193,0.031335,0.028302,3.766682,0.040626,-2.465478,3.661658,4.311326,4.431295
4,1980-06-30,3.953413,0.035451,0.609975,0.022955,0.024909,3.768565,0.033697,-2.498637,3.643353,4.207531,4.284341
...,...,...,...,...,...,...,...,...,...,...,...,...
173,2022-09-30,5.008861,0.014444,2.906448,0.006571,0.007648,5.331930,0.014947,-3.370713,4.613407,4.814593,5.240803
174,2022-12-31,5.015219,0.009122,2.961698,0.009901,0.009396,5.332679,0.011538,-3.365033,4.485170,4.719217,5.244402
175,2023-03-31,5.018379,0.010911,3.020958,0.011307,0.008954,5.343477,0.008314,-3.395617,4.397714,4.696997,5.416306
176,2023-06-30,5.029266,0.004715,3.095903,0.012372,0.008826,5.352381,0.007976,-3.403638,4.361766,4.674107,5.315188


In [7]:
# 1. This is to test the weak exogeneity for each variable components in the us_df table. 
# 2. We need to do the AIC and BIC test to find out the lags that are most optimal for the VARX



p,q are: (1,0)
p,q are: (1,1)
p,q are: (1,2)
p,q are: (2,0)
p,q are: (2,1)
p,q are: (2,2)
p,q are: (3,0)
p,q are: (3,1)
p,q are: (3,2)


In [8]:
# to run the VARX model.
# After running the weak exogeneity test, we will then proceed to run the VARX for
# the variables are do not violate the weak exogeneity test. 
endog_vars = ['y', 'Dp', 'eq', 'r', 'lr', 'poil', 'pmat', 'pmetal']
exog_vars = ['ys', 'Dps']

us_endog_df = us_df[endog_vars]
us_exog_df = us_df[exog_vars]

display(us_endog_df)
display(us_exog_df)

# lets assume that our lags are as follows p_{domestic} = 2, q_{foreign} = 1
# so we are now finding for VAR(p, q) = VAR(2, 1)
# so for our exog data we need to start at iloc[1:] as our t 
# and shift(1).iloc[1:] as our t - 1
# for endog_data we will start at us_df.iloc[2:] as our t 
# and us_df.shift(1).iloc[2:] as our t - 2
p, q = 2, 1

# to define the exog data here based on lags (defined by AIC/BIC/HQIC) and shifts (based on also lags?).
us_exog_contemp_df = us_exog_df.iloc[p:] # align to match the loss of 2 domestic lags.
us_exog_lag1_df = us_exog_df.shift(1).iloc[p:]
display(us_exog_contemp_df)
display(us_exog_lag1_df)
print(f"us_exog_df shift(1) is: {display(us_exog_df.shift(1))}")
print(f"us_exog_df shift(1).iloc[p:] is: {display(us_exog_df.shift(1).iloc[p:])}")

# Concatenate the exog data here with the shifts and lags. 
us_exog_final_df = pd.concat([us_exog_contemp_df, us_exog_lag1_df], axis = 1)
print(f"us_exog_final_df is: {display(us_exog_final_df)}")
col_names = [f"{col}_lag1" for col in us_exog_df.columns]
all_col_exog_names = us_exog_df.columns.tolist() + col_names
print(all_col_exog_names)
# This is to ensure that the naming convention for the lags and the contemporary data are defined correctly.
us_exog_final_df.columns = all_col_exog_names
print(f"us_exog_final_df is: {display(us_exog_final_df)}")

# Add in the constant from the exog data. 
us_exog_final_df = sm.add_constant(us_exog_final_df)
print(f"us_exog_final_df after including constant is: {display(us_exog_final_df)}")

# define us endogeneous data
us_endog_final_df = us_endog_df.iloc[p:]
display(us_endog_final_df)

# Fit the level VARX model via statsmodels 
# we used maxlags = max(p,q) for the domestic variables and passing foreign vectors as exog.
varx_model = VAR(endog = us_endog_final_df, exog = us_exog_final_df)
varx_results = varx_model.fit(maxlags = p)

print(f"--- US VARX({p},{q}) Model Estimated Successfully ---")
print(f"Endogenous Matrix Dimensions (Variables): {us_endog_final_df.shape[1]}")
print(f"Exogenous Matrix Dimensions (Regressors): {us_exog_final_df.shape[1]}")
print(f"\nCoefficient Parameters shape for the US system:", varx_results.params.shape)
print(varx_results.summary())

,y,Dp,eq,r,lr,poil,pmat,pmetal
0,3.960503,0.033730,0.684670,0.022407,0.021804,3.433544,4.148960,4.200168
1,3.967677,0.032477,0.699070,0.023084,0.021781,3.576322,4.272871,4.198063
2,3.970622,0.028823,0.640953,0.027982,0.024841,3.696729,4.220098,4.305733
3,3.973806,0.038217,0.615193,0.031335,0.028302,3.661658,4.311326,4.431295
4,3.953413,0.035451,0.609975,0.022955,0.024909,3.643353,4.207531,4.284341
...,...,...,...,...,...,...,...,...
173,5.008861,0.014444,2.906448,0.006571,0.007648,4.613407,4.814593,5.240803
174,5.015219,0.009122,2.961698,0.009901,0.009396,4.485170,4.719217,5.244402
175,5.018379,0.010911,3.020958,0.011307,0.008954,4.397714,4.696997,5.416306
176,5.029266,0.004715,3.095903,0.012372,0.008826,4.361766,4.674107,5.315188


,ys,Dps
0,3.723439,0.027880
1,3.736803,0.029817
2,3.755427,0.029535
3,3.766682,0.040626
4,3.768565,0.033697
...,...,...
173,5.331930,0.014947
174,5.332679,0.011538
175,5.343477,0.008314
176,5.352381,0.007976


,ys,Dps
2,3.755427,0.029535
3,3.766682,0.040626
4,3.768565,0.033697
5,3.776422,0.032422
6,3.787429,0.032043
...,...,...
173,5.331930,0.014947
174,5.332679,0.011538
175,5.343477,0.008314
176,5.352381,0.007976


,ys,Dps
2,3.736803,0.029817
3,3.755427,0.029535
4,3.766682,0.040626
5,3.768565,0.033697
6,3.776422,0.032422
...,...,...
173,5.320754,0.021158
174,5.331930,0.014947
175,5.332679,0.011538
176,5.343477,0.008314


,ys,Dps
0,NaN,NaN
1,3.723439,0.027880
2,3.736803,0.029817
3,3.755427,0.029535
4,3.766682,0.040626
...,...,...
173,5.320754,0.021158
174,5.331930,0.014947
175,5.332679,0.011538
176,5.343477,0.008314


...

,const,ys,Dps,ys_lag1,Dps_lag1
2,1.0,3.755427,0.029535,3.736803,0.029817
3,1.0,3.766682,0.040626,3.755427,0.029535
4,1.0,3.768565,0.033697,3.766682,0.040626
5,1.0,3.776422,0.032422,3.768565,0.033697
6,1.0,3.787429,0.032043,3.776422,0.032422
...,...,...,...,...,...
173,1.0,5.331930,0.014947,5.320754,0.021158
174,1.0,5.332679,0.011538,5.331930,0.014947
175,1.0,5.343477,0.008314,5.332679,0.011538
176,1.0,5.352381,0.007976,5.343477,0.008314


us_exog_final_df after including constant is: None


,y,Dp,eq,r,lr,poil,pmat,pmetal
2,3.970622,0.028823,0.640953,0.027982,0.024841,3.696729,4.220098,4.305733
3,3.973806,0.038217,0.615193,0.031335,0.028302,3.661658,4.311326,4.431295
4,3.953413,0.035451,0.609975,0.022955,0.024909,3.643353,4.207531,4.284341
5,3.951749,0.018735,0.694921,0.021896,0.025985,3.550055,4.289884,4.277811
6,3.970122,0.025836,0.724808,0.031908,0.029275,3.679337,4.284165,4.195716
...,...,...,...,...,...,...,...,...
173,5.008861,0.014444,2.906448,0.006571,0.007648,4.613407,4.814593,5.240803
174,5.015219,0.009122,2.961698,0.009901,0.009396,4.485170,4.719217,5.244402
175,5.018379,0.010911,3.020958,0.011307,0.008954,4.397714,4.696997,5.416306
176,5.029266,0.004715,3.095903,0.012372,0.008826,4.361766,4.674107,5.315188


--- US VARX(2,1) Model Estimated Successfully ---
Endogenous Matrix Dimensions (Variables): 8
Exogenous Matrix Dimensions (Regressors): 5

Coefficient Parameters shape for the US system: (22, 8)
  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Sun, 19, Jul, 2026
Time:                     22:29:30
--------------------------------------------------------------------
No. of Equations:         8.00000    BIC:                   -65.9803
Nobs:                     174.000    HQIC:                  -67.8794
Log likelihood:           4219.12    FPE:                9.16476e-31
AIC:                     -69.1757    Det(Omega_mle):     3.53563e-31
--------------------------------------------------------------------
Results for equation y
               coefficient       std. error           t-stat            prob
----------------------------------------------------------------------------
const             0.113483             

C:\Users\benchee\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\vector_ar\var_model.py:1558: RuntimeWarning: invalid value encountered in sqrt
  stderr = np.sqrt(np.diag(self.cov_params()))


# The code below is supposed to be a module to find the best lag order based on AIC, BIC and HQIC. 
# However, seeing that we need to see the results of the VECMX. I will get back to this when i can have a better picture on how the lags p, q = 2,2 works. 

In [36]:
# this code block will be used for the AIC, BIC and HQIC

# define variables
exog_df = us_exog_df
endog_df = us_endog_df
#display(exog_df)
#print(f"endog_df is: \n {display(endog_df)}")

# Grid parameters
max_p = 3 # for domestic/ endog vars
max_q = 2 # for foreign/ exog vars
max_lag = max(max_p, max_q)

results = []

# 2. High level optimization loop using statsmodels internals. 
for p in range(1, max_p + 1):
    for q in range(0, max_q + 1):
        
        print(f"p,q are: ({p},{q})")
        # Construct foreign variables lag columns
        exog_list = []
        for lag in range(0, q + 1):
            print(f"lag (q) now is: {lag}")
            target_exog_df = exog_df.shift(lag)
            if lag >= 1:
                target_exog_df.columns = [f"{col}_lag{q}" for col in target_exog_df.columns]

            exog_list.append(target_exog_df) # shift the lag based on q variable.      
        
        X_exog = pd.concat(exog_list, axis = 1).dropna() # to concatenate by columns. 
#        print(f"X_exog is:\n {display(X_exog)}")
#        print(f"X_exog.index is:\n {display(X_exog.index)}")

        # Align endog target dataframe to match exogenous shift row losses.
        Y_target = endog_df.loc[X_exog.index]
#        print(f"Y_target is:\n {display(Y_target)}")

        # CRUCIAL GVAR TRIMMING: Ensure ALL lag combinations run on the exact same time window. 
        # This keeps the comparison mathematically fair across the loops. 
        X_exog_clean = X_exog.iloc[(max_lag - p):] # max_lag = 3, p = 1, ans = 2 
        Y_target_clean = Y_target.iloc[(max_lag - p):] # max_lag = 3, p = 1, ans = 2

#        print(f"X_exog_clean is:\n {display(X_exog_clean)}")
        print(f"X_exog_clean shape is:\n {(X_exog_clean.shape)}")
#        print(f"Y_target_clean is:\n {display(Y_target_clean)}")
        print(f"Y_target_clean shape is:\n {(Y_target_clean.shape)}")

        # 3. Call the built-in statsmodels VAR function with exog parameters
        # maxlags = p forces the engine to fit exactly 'p' for domestic lags
        model = VAR(endog = Y_target_clean, exog = X_exog_clean)
        results_fitted = model.fit(maxlags=p)

        # 4. Pull the native optimized AIC/BIC/HQIC calculations from the results object.
        results.append({
            'p' : p,
            'q' : q,
            'AIC' : results_fitted.aic,
            'BIC' : results_fitted.bic,
            'HQIC' : results_fitted.hqic,
        })
        
#        if q == 1: 
#            print(f"lets look at the results first.")
#            break

# 5. Display the final score sheet.
df_scores = pd.DataFrame(results)
print("--- Statsmodels Evaluated Scores ---")
print(f"df_scores is: \n {display(df_scores)}")
print(df_scores.to_string(index=False))

# Extract the winning combinations
print(f"\nWinning Architecture by AIC: P={df_scores.loc[df_scores['AIC'].idxmin(), 'p']}, Q = {df_scores.loc[df_scores['AIC'].idxmin(), 'q']}")
print(f"\nWinning Architecture by BIC: P={df_scores.loc[df_scores['BIC'].idxmin(), 'p']}, Q = {df_scores.loc[df_scores['BIC'].idxmin(), 'q']}")
print(f"\nWinning Architecture by HQIC: P={df_scores.loc[df_scores['HQIC'].idxmin(), 'p']}, Q = {df_scores.loc[df_scores['HQIC'].idxmin(), 'q']}")

p,q are: (1,0)
lag (q) now is: 0
X_exog_clean shape is:
 (176, 2)
Y_target_clean shape is:
 (176, 8)
p,q are: (1,1)
lag (q) now is: 0
lag (q) now is: 1
X_exog_clean shape is:
 (175, 4)
Y_target_clean shape is:
 (175, 8)
p,q are: (1,2)
lag (q) now is: 0
lag (q) now is: 1
lag (q) now is: 2
X_exog_clean shape is:
 (174, 6)
Y_target_clean shape is:
 (174, 8)
p,q are: (2,0)
lag (q) now is: 0
X_exog_clean shape is:
 (177, 2)
Y_target_clean shape is:
 (177, 8)
p,q are: (2,1)
lag (q) now is: 0
lag (q) now is: 1
X_exog_clean shape is:
 (176, 4)
Y_target_clean shape is:
 (176, 8)
p,q are: (2,2)
lag (q) now is: 0
lag (q) now is: 1
lag (q) now is: 2
X_exog_clean shape is:
 (175, 6)
Y_target_clean shape is:
 (175, 8)
p,q are: (3,0)
lag (q) now is: 0
X_exog_clean shape is:
 (178, 2)
Y_target_clean shape is:
 (178, 8)
p,q are: (3,1)
lag (q) now is: 0
lag (q) now is: 1
X_exog_clean shape is:
 (177, 4)
Y_target_clean shape is:
 (177, 8)
p,q are: (3,2)
lag (q) now is: 0
lag (q) now is: 1
lag (q) now is:

,p,q,AIC,BIC,HQIC
0,1,0,-67.836095,-66.244659,-67.190563
1,1,1,-69.152206,-67.264035,-68.386248
2,1,2,-69.424765,-67.237511,-68.537409
3,2,0,-68.243021,-65.494179,-67.128013
4,2,1,-69.267610,-66.217488,-68.030293
5,2,2,-69.620710,-66.266920,-68.260098
6,3,0,-68.192103,-64.285853,-66.607616
7,3,1,-69.262275,-65.050201,-67.553599
8,3,2,-69.663047,-65.142721,-67.829178


df_scores is: 
 None
 p  q        AIC        BIC       HQIC
 1  0 -67.836095 -66.244659 -67.190563
 1  1 -69.152206 -67.264035 -68.386248
 1  2 -69.424765 -67.237511 -68.537409
 2  0 -68.243021 -65.494179 -67.128013
 2  1 -69.267610 -66.217488 -68.030293
 2  2 -69.620710 -66.266920 -68.260098
 3  0 -68.192103 -64.285853 -66.607616
 3  1 -69.262275 -65.050201 -67.553599
 3  2 -69.663047 -65.142721 -67.829178

Winning Architecture by AIC: P=3, Q = 2

Winning Architecture by BIC: P=1, Q = 1

Winning Architecture by HQIC: P=1, Q = 2
